# Chapter 10: Mechanics of Options Markets
## Replicating Exchange Rules, Margin Accounts, and Practice Questions with Python

In the world of derivatives, an option contract is a versatile financial instrument. Unlike forward contracts, where both parties are committed to a future transaction, options grant **rights** without **obligations**. 
* The **buyer** of an option pays an upfront premium to secure the right to buy (Call) or sell (Put) an underlying asset.
* The **writer (seller)** of an option receives the premium and takes on the potential obligation, which requires maintaining a **margin account** to manage default risk.

In this notebook, we explore the mechanical and regulatory rules that govern exchange-traded options as outlined in **Chapter 10** of John Hull's *Options, Futures, and Other Derivatives*:
1. **Naked Options Margin Requirements**: Replicating CBOE rules for written naked options.
2. **Replication of Example 10.3**: Modeling margin calculations for written calls and puts.
3. **Practice Questions**: Solving and programming the math for **Practice Questions 10.1 to 10.22** with solutions positioned directly underneath each question.


---
## 1. CBOE Margin Requirements for Naked Options

A **naked (uncovered) option** is a written option where the writer does not own the underlying stock (in the case of a call) or does not hold cash to fully secure the position (in the case of a put). Because the writer's risk is theoretically unlimited (for calls) or substantial (for puts), the Chicago Board Options Exchange (CBOE) enforces strict margin requirements.

### Written Naked Call Options
The initial and maintenance margin required per contract (100 shares) is the **greater** of the following two calculations:
1. **Calculation 1**: $100\%$ of the option proceeds (current premium) plus $20\%$ of the underlying stock price, less the out-of-the-money (OTM) amount:
   $$\text{Margin}_1 = 100 \times \left( c + 0.20 \times S_t - \max(0, K - S_t) \right)$$
2. **Calculation 2**: $100\%$ of the option proceeds plus $10\%$ of the underlying stock price:
   $$\text{Margin}_2 = 100 \times \left( c + 0.10 \times S_t \right)$$

---

### Written Naked Put Options
Similarly, the margin required per written put contract (100 shares) is the **greater** of:
1. **Calculation 1**: $100\%$ of the option proceeds plus $20\%$ of the underlying stock price, less the out-of-the-money (OTM) amount:
   $$\text{Margin}_1 = 100 \times \left( p + 0.20 \times S_t - \max(0, S_t - K) \right)$$
2. **Calculation 2**: $100\%$ of the option proceeds plus $10\%$ of the strike price:
   $$\text{Margin}_2 = 100 \times \left( p + 0.10 \times K \right)$$

---

### Special Adjustments:
* **Broad-Based Index Options**: The $20\%$ multiplier in the first calculation is replaced by **$15\%$** because index volatility is generally lower than individual stock volatility.
* **Option Buyers**: Buyers face no margin requirements because their maximum potential loss is capped at the premium already paid.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Format decimals to 2-decimal cash precision
pd.options.display.float_format = '{:,.2f}'.format

# Sleek design palette
navy_blue = '#1E3A8A'
emerald = '#10B981'
crimson = '#EF4444'
amber = '#F59E0B'
violet = '#8B5CF6'
dark_gray = '#374151'

# CBOE Naked Options Margin Calculator
def calculate_naked_margin(option_type, s, k, premium, n_contracts=1, index_option=False):
    multiplier = 0.15 if index_option else 0.20
    n_shares = n_contracts * 100
    
    if option_type.lower() == 'call':
        otm_amount = max(0, k - s)
        calc1 = n_shares * (premium + multiplier * s - otm_amount)
        calc2 = n_shares * (premium + 0.10 * s)
    elif option_type.lower() == 'put':
        otm_amount = max(0, s - k)
        calc1 = n_shares * (premium + multiplier * s - otm_amount)
        calc2 = n_shares * (premium + 0.10 * k)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
        
    return max(calc1, calc2), calc1, calc2


In [ ]:
# --- Example 10.3 Replication ---
# 4 Naked Call Options: Option price = $5, Strike = $40, Stock = $38
margin_call, c1_call, c2_call = calculate_naked_margin('call', s=38.0, k=40.0, premium=5.0, n_contracts=4)

# 4 Naked Put Options (same variables, stock is $2 ITM)
margin_put, c1_put, c2_put = calculate_naked_margin('put', s=38.0, k=40.0, premium=5.0, n_contracts=4)

print("Example 10.3 Replications:")
print(f"  Naked Call Margin Required: ${margin_call:,.2f} (Calc 1: ${c1_call:.2f}, Calc 2: ${c2_call:.2f})")
print(f"  Naked Put Margin Required:  ${margin_put:,.2f} (Calc 1: ${c1_put:.2f}, Calc 2: ${c2_put:.2f})")


In [ ]:
# --- Plotting Example 10.3 Call vs. Put Margin Requirements dynamically ---
s_range_ex = np.linspace(20, 60, 300)
c_margins_ex = []
p_margins_ex = []

for s in s_range_ex:
    c_margins_ex.append(calculate_naked_margin('call', s, k=40.0, premium=5.0, n_contracts=4)[0])
    p_margins_ex.append(calculate_naked_margin('put', s, k=40.0, premium=5.0, n_contracts=4)[0])

plt.figure(figsize=(9, 5), dpi=100)
plt.plot(s_range_ex, c_margins_ex, color=crimson, linewidth=2.5, label='Naked Call Margin (Strike = $40)')
plt.plot(s_range_ex, p_margins_ex, color=navy_blue, linewidth=2.5, label='Naked Put Margin (Strike = $40)')
plt.axvline(38.0, color=emerald, linestyle=':', linewidth=2, label='Current Stock Price = $38')

plt.title('Example 10.3: Call vs. Put Naked Margin Requirements Profile', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Underlying Stock Price ($S_t$)', fontsize=11)
plt.ylabel('Margin Requirement ($)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10)
plt.show()


---
# Part 2: Chapter 10 Practice Questions & Python Solutions

Let's dive into the **Practice Questions 10.1 to 10.22** to master the options mechanics, taxation planning, stock split adjustments, and margin dynamics. Each question's complete solution and corresponding visualization is placed directly below the question for immediate readability.


---
## Practice Question 10.1

### Question
An investor buys a European put on a share for $3. The stock price is $42 and the strike price is $40. Under what circumstances does the investor make a profit? Under what circumstances will the option be exercised? Draw a diagram showing the variation of the investor's profit with the stock price at the maturity of the option.

### Solution
* **Option payoff at maturity**: $\max(0, 40 - S_T)$
* **Net Profit**: $\max(0, 40 - S_T) - 3$
* **Exercise Condition**: The option is exercised when it has positive intrinsic value at maturity, which occurs when the stock price is below the strike price: **$S_T < \$40$**.
* **Profit Condition**: The investor makes a net profit when the payoff exceeds the premium paid, which occurs when the stock price falls below the strike price minus the put premium: $K - p = 40 - 3 = \mathbf{\$37}$ at maturity.


In [ ]:
# --- Plotting Profit curve for Put Buyer (Question 10.1) ---
st_range = np.linspace(20, 60, 200)

payoff_put = np.maximum(0, 40 - st_range)
profit_put = payoff_put - 3.0

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range, profit_put, color=navy_blue, linewidth=2.5, label='Net Profit')
plt.plot(st_range, payoff_put, color=emerald, linewidth=1.5, linestyle='--', label='Payoff at Maturity')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(37, color=crimson, linestyle=':', label='Breakeven ($37)')

plt.title('10.1: Long European Put (Buyer) Profit Profile', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Profit / Loss ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## Practice Question 10.2

### Question
An investor sells a European call on a share for $4. The stock price is $47 and the strike price is $50. Under what circumstances does the investor make a profit? Under what circumstances will the option be exercised? Draw a diagram showing the variation of the investor's profit with the stock price at the maturity of the option.

### Solution
* **Option payoff at maturity**: $-\max(0, S_T - 50)$
* **Net Profit**: $4 - \max(0, S_T - 50)$
* **Exercise Condition**: The option is exercised by the holder (the buyer) when it is in-the-money at maturity: **$S_T > \$50$**.
* **Profit Condition**: The writer (the seller) makes a net profit when the stock price remains below the strike price plus the call premium: $K + c = 50 + 4 = \mathbf{\$54}$ at maturity.


In [ ]:
# --- Plotting Profit curve for Call Writer (Question 10.2) ---
st_range = np.linspace(30, 70, 200)

payoff_call = -np.maximum(0, st_range - 50)
profit_call = payoff_call + 4.0

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range, profit_call, color=crimson, linewidth=2.5, label='Net Profit')
plt.plot(st_range, payoff_call, color=navy_blue, linewidth=1.5, linestyle='--', label='Payoff at Maturity')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(54, color=emerald, linestyle=':', label='Breakeven ($54)')

plt.title('10.2: Short European Call (Writer) Profit Profile', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Profit / Loss ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## Practice Question 10.3

### Question
An investor sells a European call option with strike price of $K$ and maturity $T$ and buys a put with the same strike price and maturity. Describe the investor’s position.

### Solution
* **Short Call Payoff**: $-\max(0, S_T - K)$
* **Long Put Payoff**: $+\max(0, K - S_T)$
* **Net Payoff**: 
  $$\text{Payoff} = \max(0, K - S_T) - \max(0, S_T - K)$$
  * If $S_T > K$: Payoff $= 0 - (S_T - K) = K - S_T$.
  * If $S_T \le K$: Payoff $= (K - S_T) - 0 = K - S_T$.
* The net payoff is **$K - S_T$** regardless of where the stock price lands.
* This is exactly the same payoff profile as a **short position in a forward contract** with delivery price $K$ and delivery date $T$.


In [ ]:
# --- Plotting Question 10.3: Synthetic Short Forward ---
st_range_3 = np.linspace(20, 60, 200)
strike_k = 40.0

short_call_payoff = -np.maximum(0, st_range_3 - strike_k)
long_put_payoff = np.maximum(0, strike_k - st_range_3)
synthetic_fwd = short_call_payoff + long_put_payoff

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range_3, short_call_payoff, color=crimson, linestyle='--', linewidth=1.5, label='Short Call Payoff')
plt.plot(st_range_3, long_put_payoff, color=emerald, linestyle='--', linewidth=1.5, label='Long Put Payoff')
plt.plot(st_range_3, synthetic_fwd, color=navy_blue, linewidth=3, label='Synthetic Short Forward (Net Payoff)')

plt.axhline(0, color='black', linewidth=1)
plt.axvline(strike_k, color=dark_gray, linestyle=':', alpha=0.7)

plt.title('Question 10.3: Replicating a Short Forward synthetically', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Payoff ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10)
plt.show()


---
## Practice Question 10.4

### Question
A company declares a 2-for-1 stock split. Explain how the terms change for a call option with a strike price of $60$.

### Solution
* A 2-for-1 stock split doubles the number of shares and halves the share price.
* To preserve the economic value for both option holders and writers:
  * The number of shares covered by the option is **doubled**: from $100$ shares to **$200$ shares**.
  * The strike price is **halved**: from $\$60$ to **$\$30$**.
* Note that the total commitment remains identical: $100 \times \$60 = 200 \times \$30 = \$6,000$.


---
## Practice Question 10.5

### Question
“Employee stock options issued by a company are different from regular exchange-traded call options on the company’s stock because they can affect the capital structure of the company.” Explain this statement.

### Solution
1. **Share Creation & Dilution**:
   * When an **exchange-traded** call option is exercised, the option holder buys existing shares from the option writer. The underlying company is not involved in any way, and the total number of shares outstanding remains completely unchanged.
   * When an **employee stock option** is exercised, the company issues **new shares** of stock and sells them to the employee for the strike price. This adds new shares to the market, causing **dilution** of equity and directly altering the firm's capital structure (specifically increasing common equity and total shares outstanding).
2. **Transferability & Trading**:
   * Exchange-traded options are standardized, liquid, and can be sold to anyone at any time.
   * Employee stock options are non-transferable (cannot be sold to anyone else) and are subject to **vesting periods** (the employee must work for the company for a certain time before the options can be exercised).


---
## Practice Question 10.6

### Question
Suppose that a European call option to buy a share for $100.00 costs $5.00 and is held until maturity. Under what circumstances will the holder of the option make a profit? Under what circumstances will the option be exercised? Draw a diagram illustrating how the profit from a long position in the option depends on the stock price at maturity of the option.

### Solution
* **Exercise Condition**: The option is exercised when it is in-the-money at maturity: **$S_T > \$100$**.
* **Profit Condition**: The investor makes a net profit when the stock price climbs above the strike plus the premium: $K + c = 100 + 5 = \mathbf{\$105}$ at maturity.


In [ ]:
# --- Plotting Profit curve for Call Buyer (Question 10.6) ---
st_range_6 = np.linspace(60, 140, 200)

payoff_call_6 = np.maximum(0, st_range_6 - 100)
profit_call_6 = payoff_call_6 - 5.0

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range_6, profit_call_6, color=emerald, linewidth=2.5, label='Net Profit')
plt.plot(st_range_6, payoff_call_6, color=navy_blue, linewidth=1.5, linestyle='--', label='Payoff at Maturity')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(105, color=crimson, linestyle=':', label='Breakeven ($105)')

plt.title('10.6: Long European Call (Buyer) Profit Profile', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Profit / Loss ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## Practice Question 10.7

### Question
Suppose that a European put option to sell a share for $60 costs $8 and is held until maturity. Under what circumstances will the seller of the option (the party with the short position) make a profit? Under what circumstances will the option be exercised? Draw a diagram illustrating how the profit from a short position in the option depends on the stock price at maturity of the option.

### Solution
* **Exercise Condition**: The option is exercised by the holder (buyer) when it is in-the-money at maturity: **$S_T < \$60$**.
* **Profit Condition**: The writer makes a net profit when the stock price remains above the strike minus the premium: $K - p = 60 - 8 = \mathbf{\$52}$ at maturity.


In [ ]:
# --- Plotting Profit curve for Put Writer (Question 10.7) ---
st_range_7 = np.linspace(30, 90, 200)

payoff_put_7 = -np.maximum(0, 60 - st_range_7)
profit_put_7 = payoff_put_7 + 8.0

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range_7, profit_put_7, color=navy_blue, linewidth=2.5, label='Net Profit')
plt.plot(st_range_7, payoff_put_7, color=crimson, linewidth=1.5, linestyle='--', label='Payoff at Maturity')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(52, color=emerald, linestyle=':', label='Breakeven ($52)')

plt.title('10.7: Short European Put (Writer) Profit Profile', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Profit / Loss ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## Practice Question 10.8

### Question
Consider the following portfolio: a newly entered-into long forward contract on an asset and a long position in a European put option on the asset with the same maturity as the forward contract and a strike price that is equal to the forward price of the asset at the time the portfolio is set up. Show that it has the same value as a European call option with the same strike price and maturity as the European put option. Deduce that a European put option has the same value as a European call option with the same strike price and maturity when the strike price for both options is the forward price.

### Solution
* **Payoff of Long Forward** at maturity: $S_T - F_0$
* **Payoff of Long Put** (strike $K = F_0$): $\max(0, F_0 - S_T)$
* **Net Payoff of the portfolio**:
  $$\text{Net Payoff} = (S_T - F_0) + \max(0, F_0 - S_T)$$
  * If $S_T > F_0$: Payoff $= S_T - F_0 + 0 = S_T - F_0$.
  * If $S_T \le F_0$: Payoff $= S_T - F_0 + F_0 - S_T = 0$.
* This matches exactly the payoff of a **Long European Call Option** with strike $K = F_0$, which is $\max(0, S_T - F_0)$.
* Since the payoffs are identical at maturity, their initial values must be identical to prevent arbitrage:
  $$V_{\text{forward}} + P = C$$
* Because a newly entered forward contract has an initial value of $0$, we get:
  $$0 + P = C \implies P = C$$
* Thus, European call and put options have the **exact same price** when their strike is the forward price.


In [ ]:
# --- Plotting Question 10.8: Synthetic Long Call ---
st_range_8 = np.linspace(20, 60, 200)
fwd_price = 40.0

long_fwd_payoff = st_range_8 - fwd_price
long_put_payoff_8 = np.maximum(0, fwd_price - st_range_8)
synthetic_call = long_fwd_payoff + long_put_payoff_8

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range_8, long_fwd_payoff, color=crimson, linestyle='--', linewidth=1.5, label='Long Forward Payoff')
plt.plot(st_range_8, long_put_payoff_8, color=emerald, linestyle='--', linewidth=1.5, label='Long Put Payoff')
plt.plot(st_range_8, synthetic_call, color=navy_blue, linewidth=3, label='Synthetic Long Call (Net Payoff)')

plt.axhline(0, color='black', linewidth=1)
plt.axvline(fwd_price, color=dark_gray, linestyle=':', alpha=0.7)

plt.title('Question 10.8: Replicating a Long Call synthetically', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Payoff ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10)
plt.show()


---
## Practice Question 10.9

### Question
A trader buys a call option with a strike price of $45 and a put option with a strike price of $40. Both options have the same maturity. The call costs $3 and the put costs $4. Draw a diagram showing the variation of the trader’s profit with the asset price.

### Solution
* **Upfront cost paid** $= 3 + 4 = \$7$.
* **Payoff at maturity**: $\max(0, S_T - 45) + \max(0, 40 - S_T)$.
* **Break-even points**:
  * Lower: $40 - 7 = \mathbf{\$33}$
  * Upper: $45 + 7 = \mathbf{\$52}$


In [ ]:
# --- Plotting Profit curve for a Strangle (Question 10.9) ---
st_range_9 = np.linspace(20, 65, 200)

payoff_strangle = np.maximum(0, st_range_9 - 45) + np.maximum(0, 40 - st_range_9)
profit_strangle = payoff_strangle - 7.0

plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(st_range_9, profit_strangle, color=violet, linewidth=2.5, label='Net Profit')
plt.plot(st_range_9, payoff_strangle, color='#9CA3AF', linewidth=1.5, linestyle='--', label='Payoff')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(33, color=crimson, linestyle=':', label='Lower Breakeven ($33)')
plt.axvline(52, color=emerald, linestyle=':', label='Upper Breakeven ($52)')

plt.title('10.9: Strangle Option Strategy Profit Profile', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price at Maturity ($S_T$)', fontsize=10)
plt.ylabel('Profit / Loss ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## Practice Question 10.10

### Question
Explain why an American option is always worth at least as much as a European option on the same asset with the same strike price and exercise date.

### Solution
* An American option contains all the rights of a European option (the right to hold and exercise at maturity $T$).
* In addition, it contains the **extra right** of early exercise at any time $t \le T$.
* Since an American option offers more flexibility than a European option, its market value must be greater than or equal to the European option.


---
## Practice Question 10.11

### Question
Explain why an American option is always worth at least as much as its intrinsic value.

### Solution
* If an American option traded below its intrinsic value (immediate exercise payoff, like $S - K$ for a call), an arbitrageur would immediately buy the option, exercise it immediately, and make a risk-free profit.
* Because the option can be exercised *instantly*, its price cannot fall below the cash payoff of immediate exercise.


---
## Practice Question 10.12

### Question
The treasurer of a corporation is trying to choose between options and forward contracts to hedge the corporation’s foreign exchange risk. Discuss the advantages and disadvantages of each.

### Solution
* **Forward Contracts**:
  * *Advantages*: No upfront cost (premium is 0). Completely locks in the future exchange rate, removing all variance.
  * *Disadvantages*: Completely eliminates any positive upside. If the exchange rate moves in the company's favor, they are still locked into the forward rate.
* **Option Contracts**:
  * *Advantages*: Functions like insurance. Limits the downside risk but allows the company to participate in any favorable exchange rate movements (by letting the option expire unexercised).
  * *Disadvantages*: Requires paying a non-refundable upfront cash premium.


---
## Practice Question 10.13

### Question
Consider an exchange-traded call option contract to buy 500 shares with a strike price of $40 and maturity in 4 months. Explain how the terms of the option contract change when there is: (a) a 10% stock dividend; (b) a 10% cash dividend; and (c) a 4-for-1 stock split.

### Solution
* **(a) 10% Stock Dividend**:
  * The number of shares increases by $10\%$: $500 \times 1.10 = \mathbf{550}$ shares.
  * The strike price is adjusted downward: $40 / 1.10 = \mathbf{\$36.36}$.
* **(b) 10% Cash Dividend**:
  * Standard exchange-traded options are **not adjusted** for cash dividends. No change occurs.
* **(c) 4-for-1 Stock Split**:
  * The number of shares increases by a factor of 4: $500 \times 4 = \mathbf{2,000}$ shares.
  * The strike price is quartered: $40 / 4 = \mathbf{\$10.00}$.


---
## Practice Question 10.14

### Question
“If most of the call options on a stock are in the money, it is likely that the stock price has risen rapidly in the last few months.” Discuss this statement.

### Solution
* The statement is **True**. Options are launched with strike prices set around the current market price of the stock.
* Call options become in-the-money when $S > K$. If a high percentage of outstanding call options are ITM, the stock price must have recently risen rapidly above those pre-existing strike prices.


---
## Practice Question 10.15

### Question
What is the effect of an unexpected cash dividend on (a) a call option price and (b) a put option price?

### Solution
* When a stock goes ex-dividend, its share price drops by approximately the cash dividend amount.
* **(a) Call price**: The stock price drop **decreases** the value of call options.
* **(b) Put price**: The stock price drop **increases** the value of put options.


---
## Practice Question 10.16

### Question
Options on a stock are on a March, June, September, and December cycle. What options trade on (a) March 1, (b) June 30, and (c) August 5?

### Solution
* The standard rule is that options trade for the current month, the next month, and the next two months in the cycle:
  * **(a) March 1**: March, April, June, September.
  * **(b) June 30**: July, August, September, December.
  * **(c) August 5**: August, September, December, March.


---
## Practice Question 10.17

### Question
Explain why the market maker’s bid–ask spread represents a real cost to options investors.

### Solution
* The bid-ask spread is the difference between the price at which the market maker is willing to sell (ask) and buy (bid).
* An investor who wants to enter a position immediately buys at the higher ask, and if they close it out immediately, they sell at the lower bid.
* This spread is a direct transactional drag (cost) paid by the investor to the market maker for providing liquidity.


---
## Practice Question 10.18

### Question
A U.S. investor writes five naked call option contracts. The option price is $3.50, the strike price is $60.00, and the stock price is $57.00. What is the initial margin requirement?

### Solution
* Number of options $= 5 \times 100 = 500$ options.
* Strike price $K = 60.00$, Stock price $S_t = 57.00$, Premium $c = 3.50$.
* The option is out-of-the-money by $60.00 - 57.00 = \$3.00$.
* **Calculation 1**:
  $$\text{Margin}_1 = 500 \times (3.50 + 0.20 \times 57.00 - 3.00) = 500 \times (3.50 + 11.40 - 3.00) = \$5,950$$
* **Calculation 2**:
  $$\text{Margin}_2 = 500 \times (3.50 + 0.10 \times 57.00) = 500 \times (3.50 + 5.70) = \$4,600$$
* The margin required is the greater of the two: **$\$5,950$**.


In [ ]:
# --- Question 10.18 calculations ---
m, calc1, calc2 = calculate_naked_margin('call', s=57.0, k=60.0, premium=3.50, n_contracts=5)
print(f"10.18 Naked Call Margin: ${m:,.2f} (Calc 1: ${calc1:.2f}, Calc 2: ${calc2:.2f})\n")

# --- Plotting Question 10.18 Naked Call Margin Profile dynamically ---
s_range_18 = np.linspace(40, 80, 200)
m_std_18, m1_18, m2_18 = [], [], []

for s in s_range_18:
    m, c1, c2 = calculate_naked_margin('call', s, k=60.0, premium=3.50, n_contracts=5)
    m_std_18.append(m)
    m1_18.append(c1)
    m2_18.append(c2)

plt.figure(figsize=(9, 5), dpi=100)
plt.plot(s_range_18, m_std_18, color=navy_blue, linewidth=3, label='Naked Call Margin Required')
plt.plot(s_range_18, m1_18, color=crimson, linestyle=':', label='Calc 1 (Proceeds + 20% Spot - OTM)')
plt.plot(s_range_18, m2_18, color=amber, linestyle='-.', label='Calc 2 (Proceeds + 10% Spot)')
plt.axvline(57.0, color=emerald, linestyle='--', linewidth=1.5, label='Stock Price = $57 (Question 10.18)')

plt.title('Question 10.18: Naked Call Margin Requirement Profile (5 Contracts)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Stock Price ($S_t$)', fontsize=10)
plt.ylabel('Margin Requirement ($)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10)
plt.show()


---
## Practice Question 10.19

### Question
Calculate the intrinsic value and time value from the mid market (average of bid and ask) prices for the September call options in Table 1.2. Do the same for the September put options in Table 1.3. Assume in each case that the current mid market stock price is $316.00.

### Solution
* **Call Intrinsic Value**: $\max(0, S - K)$.
* **Put Intrinsic Value**: $\max(0, K - S)$.
* **Time Value**: $\text{Option Price} - \text{Intrinsic Value}$.
* Let's use Python and Pandas to calculate and present these values systematically:


In [ ]:
# --- Question 10.19 Calculations ---
# Assuming September options with various strikes (300, 310, 320) and premiums
strikes = np.array([300.0, 310.0, 320.0])
call_prices = np.array([21.0, 13.5, 7.8])
put_prices = np.array([4.5, 7.2, 11.4])
s_mid = 316.0

# Intrinsic values
call_intrinsic = np.maximum(0, s_mid - strikes)
put_intrinsic = np.maximum(0, strikes - s_mid)

# Time values
call_time = call_prices - call_intrinsic
put_time = put_prices - put_intrinsic

df_19 = pd.DataFrame({
    'Strike Price ($)': strikes,
    'Call Price ($)': call_prices,
    'Call Intrinsic ($)': call_intrinsic,
    'Call Time Value ($)': call_time,
    'Put Price ($)': put_prices,
    'Put Intrinsic ($)': put_intrinsic,
    'Put Time Value ($)': put_time
})
df_19.set_index('Strike Price ($)', inplace=True)
df_19


---
## Practice Question 10.20

### Question
A trader writes 5 naked put option contracts with each contract being on 100 shares. The option price is $10, the time to maturity is 6 months, and the strike price is $64.
(a) What is the margin requirement if the stock price is $58?
(b) How would the answer to (a) change if the rules for index options applied?
(c) How would the answer to (a) change if the stock price were $70?
(d) How would the answer to (a) change if the trader is buying instead of selling the options?

### Solution
* **(a) Stock Price = $58 (In-the-Money)**:
  * Out-of-the-money amount is $0$ (since $S < K$ for a put).
  * Calculation 1:
    $$\text{Margin}_1 = 500 \times (10 + 0.20 \times 58 - 0) = 500 \times (10 + 11.6) = \$10,800$$
  * Calculation 2:
    $$\text{Margin}_2 = 500 \times (10 + 0.10 \times 64) = 500 \times (10 + 6.4) = \$8,200$$
  * Required Margin: **$\$10,800$**.
* **(b) Index Option Rules (15% multiplier)**:
  * Calculation 1:
    $$\text{Margin}_1 = 500 \times (10 + 0.15 \times 58 - 0) = 500 \times (10 + 8.7) = \$9,350$$
  * Calculation 2:
    $$\text{Margin}_2 = 500 \times (10 + 0.10 \times 64) = 500 \times (10 + 6.4) = \$8,200$$
  * Required Margin: **$\$9,350$**.
* **(c) Stock Price = $70 (Out-of-the-Money)**:
  * Out-of-the-money amount $= 70 - 64 = \$6.00$.
  * Calculation 1:
    $$\text{Margin}_1 = 500 \times (10 + 0.20 \times 70 - 6.00) = 500 \times (10 + 14.00 - 6.00) = \$9,000$$
  * Calculation 2:
    $$\text{Margin}_2 = 500 \times (10 + 0.10 \times 64) = 500 \times (10 + 6.4) = \$8,200$$
  * Required Margin: **$\$9,000$**.
* **(d) Buying Instead of Selling**:
  * Option buyers face **no margin requirements** (their risk is fully capped by the paid premium).
  * Margin Required: **$\$0.00$** (the $\$10.00$ premium is paid in full upfront).


In [ ]:
# --- Question 10.20 Calculations ---
ma, ca1, ca2 = calculate_naked_margin('put', s=58.0, k=64.0, premium=10.0, n_contracts=5)
mb, cb1, cb2 = calculate_naked_margin('put', s=58.0, k=64.0, premium=10.0, n_contracts=5, index_option=True)
mc, cc1, cc2 = calculate_naked_margin('put', s=70.0, k=64.0, premium=10.0, n_contracts=5)

print("Question 10.20 Margins:")
print(f"  (a) Stock Price $58:       ${ma:,.2f}")
print(f"  (b) Index Option Rules:    ${mb:,.2f}")
print(f"  (c) Stock Price $70:       ${mc:,.2f}")
print(f"  (d) Buying Put Option:     $0.00")


In [ ]:
# --- Plotting Naked Put Margin Stock vs Index Rules (Question 10.20) ---
s_range = np.linspace(30, 90, 300)
strike_p = 64.0
prem_p = 10.0

# 1. Stock option rules (20%)
m_std = []
m1_std = []
m2_std = []
for s in s_range:
    m, c1, c2 = calculate_naked_margin('put', s, strike_p, prem_p, n_contracts=5)
    m_std.append(m)
    m1_std.append(c1)
    m2_std.append(c2)

# 2. Index option rules (15%)
m_idx = []
for s in s_range:
    m, _, _ = calculate_naked_margin('put', s, strike_p, prem_p, n_contracts=5, index_option=True)
    m_idx.append(m)

plt.figure(figsize=(10, 6), dpi=100)

plt.plot(s_range, m_std, color=navy_blue, linewidth=3, label='CBOE Naked Put Margin (Stock)')
plt.plot(s_range, m_idx, color=emerald, linewidth=2.5, linestyle='--', label='CBOE Naked Put Margin (Index)')
plt.plot(s_range, m1_std, color=crimson, linewidth=1.5, linestyle=':', label='Calc 1 (Proceeds + 20% Spot - OTM)')
plt.plot(s_range, m2_std, color=amber, linewidth=1.5, linestyle='-.', label='Calc 2 (Proceeds + 10% Strike)')

# Add vertical line at Strike Price
plt.axvline(strike_p, color='#4B5563', linestyle='-', alpha=0.5)
plt.text(strike_p + 1, 12000, 'Strike Price = $64', fontsize=10, fontweight='bold', color='#4B5563')

plt.title('CBOE Naked Put Margin Requirement Profile (5 Contracts)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Underlying Stock Price ($S_t$)', fontsize=11)
plt.ylabel('Margin Requirement ($)', fontsize=11)
plt.ylim(7000, 16000)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10, facecolor='#F9FAFB', edgecolor='#E5E7EB')
plt.show()


---
## Practice Question 10.21

### Question
“If most of the call options on a stock are in the money, it is likely that the stock price has risen rapidly in the last few months.” Discuss this statement.

### Solution
* **Executive Option Critique & Indexed Options**:
  * **The Problem**: Standard employee stock options are at-the-money at issue. If the overall stock market goes up, the company's stock will likely go up regardless of executive performance. This rewards market-wide factors (Beta) rather than firm outperformance (Alpha).
  * **The Solution**: **Relative Performance Options** (Indexed Options). 
    * The option strike price is adjusted dynamically based on the performance of a competitor index or a broad market index.
    * For example, the strike price might rise by the return of the S&P 500. Under this structure, executives only gain if their company's stock outpaces competitors.


---
## Practice Question 10.22

### Question
On July 20, 2004, Microsoft surprised the market by announcing a $3 dividend. The ex-dividend date was November 17, 2004, and the payment date was December 2, 2004. Its stock price at the time was about $28. It also changed the terms of its employee stock options so that each exercise price was adjusted downward to
$$\text{Predividend exercise price} \times \frac{\text{Closing price} - \$3.00}{\text{Closing price}}$$

The number of shares covered by each stock option outstanding was adjusted upward to
$$\text{Number of shares predividend} \times \frac{\text{Closing price}}{\text{Closing price} - \$3.00}$$

“Closing Price” means the official NASDAQ closing price of a share of Microsoft common stock on the last trading day before the ex-dividend date. Evaluate this adjustment.

### Solution
* **Microsoft Special Dividend Adjustment**:
  * Standard exchange-traded options are not adjusted for cash dividends. However, Microsoft's announcement was a massive **one-time special dividend of $\$3$** (representing about $11\%$ of the stock price).
  * When the stock went ex-dividend, the share price dropped by $\$3$, which would have unfairly penalized option holders (calls became less valuable) and unfairly rewarded option writers.
  * The adjustment ratio $\frac{\text{Closing Price} - \$3.00}{\text{Closing price}}$ reflects the proportion of value retained in the stock.
  * By scaling the strike price down and scaling the number of shares up by this exact ratio, the total commitment and option value remain **completely neutral**, ensuring absolute fairness to all parties.
